In [1]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For modeling later
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# modeling used
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Display settings
pd.set_option('display.max_columns', None)


# Using XGBoost
Based off our findings, the XGBoost model seems to be the most accurate for pregame predictions.

In [2]:
df_all = pd.read_csv("../data/processed/processed_teams.csv")
df_all.head()

,team,season,name,gameId,playerTeam,opposingTeam,home_or_away,gameDate,position,situation,xGoalsPercentage,corsiPercentage,fenwickPercentage,iceTime,xOnGoalFor,xGoalsFor,xReboundsFor,xFreezeFor,xPlayStoppedFor,xPlayContinuedInZoneFor,xPlayContinuedOutsideZoneFor,flurryAdjustedxGoalsFor,scoreVenueAdjustedxGoalsFor,flurryScoreVenueAdjustedxGoalsFor,shotsOnGoalFor,missedShotsFor,blockedShotAttemptsFor,shotAttemptsFor,goalsFor,reboundsFor,reboundGoalsFor,freezeFor,playStoppedFor,playContinuedInZoneFor,playContinuedOutsideZoneFor,savedShotsOnGoalFor,savedUnblockedShotAttemptsFor,penaltiesFor,penalityMinutesFor,faceOffsWonFor,hitsFor,takeawaysFor,giveawaysFor,lowDangerShotsFor,mediumDangerShotsFor,highDangerShotsFor,lowDangerxGoalsFor,mediumDangerxGoalsFor,highDangerxGoalsFor,lowDangerGoalsFor,mediumDangerGoalsFor,highDangerGoalsFor,scoreAdjustedShotsAttemptsFor,unblockedShotAttemptsFor,scoreAdjustedUnblockedShotAttemptsFor,dZoneGiveawaysFor,xGoalsFromxReboundsOfShotsFor,xGoalsFromActualReboundsOfShotsFor,reboundxGoalsFor,totalShotCreditFor,scoreAdjustedTotalShotCreditFor,scoreFlurryAdjustedTotalShotCreditFor,xOnGoalAgainst,xGoalsAgainst,xReboundsAgainst,xFreezeAgainst,xPlayStoppedAgainst,xPlayContinuedInZoneAgainst,xPlayContinuedOutsideZoneAgainst,flurryAdjustedxGoalsAgainst,scoreVenueAdjustedxGoalsAgainst,flurryScoreVenueAdjustedxGoalsAgainst,shotsOnGoalAgainst,missedShotsAgainst,blockedShotAttemptsAgainst,shotAttemptsAgainst,goalsAgainst,reboundsAgainst,reboundGoalsAgainst,freezeAgainst,playStoppedAgainst,playContinuedInZoneAgainst,playContinuedOutsideZoneAgainst,savedShotsOnGoalAgainst,savedUnblockedShotAttemptsAgainst,penaltiesAgainst,penalityMinutesAgainst,faceOffsWonAgainst,hitsAgainst,takeawaysAgainst,giveawaysAgainst,lowDangerShotsAgainst,mediumDangerShotsAgainst,highDangerShotsAgainst,lowDangerxGoalsAgainst,mediumDangerxGoalsAgainst,highDangerxGoalsAgainst,lowDangerGoalsAgainst,mediumDangerGoalsAgainst,highDangerGoalsAgainst,scoreAdjustedShotsAttemptsAgainst,unblockedShotAttemptsAgainst,scoreAdjustedUnblockedShotAttemptsAgainst,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,playoffGame,win,shot_diff,penalty_diff,shotAttempt_diff,faceOff_diff
0,NYR,2022,NYR,2022020003,NYR,TBL,HOME,20221011,Team Level,all,0.6699,0.6068,0.5904,3600.0,36.626,5.589,3.084,7.843,1.190,18.568,12.727,4.959,5.524,4.897,39.0,10.0,22.0,71.0,3.0,6.0,0.0,2.0,3.0,24.0,11.0,36.0,46.0,6.0,12.0,30.0,24.0,19.0,17.0,26.0,14.0,9.0,1.008,1.556,3.025,1.0,2.0,0.0,70.854,49.0,48.988,5.0,0.688,1.358,1.358,4.919,4.868,4.578,25.294,2.754,1.868,5.870,1.073,14.067,8.368,2.446,2.813,2.504,26.0,8.0,12.0,46.0,1.0,2.0,0.0,6.0,0.0,10.0,15.0,25.0,33.0,4.0,8.0,17.0,19.0,8.0,15.0,24.0,6.0,4.0,0.635,0.757,1.362,0.0,0.0,1.0,46.551,34.0,34.315,5.0,0.465,0.176,0.176,3.043,3.102,2.754,0,1,13.0,2.0,25.0,13.0
1,NYR,2022,NYR,2022020017,NYR,MIN,AWAY,20221013,Team Level,all,0.4152,0.4655,0.4556,3600.0,30.783,3.165,2.387,7.092,0.930,16.263,11.163,3.123,3.400,3.356,35.0,6.0,13.0,54.0,7.0,4.0,1.0,6.0,0.0,10.0,14.0,28.0,34.0,7.0,17.0,29.0,27.0,4.0,1.0,29.0,9.0,3.0,0.641,1.088,1.436,1.0,3.0,3.0,61.002,41.0,45.018,0.0,0.536,0.336,0.336,3.365,3.596,3.554,35.567,4.457,2.568,7.138,1.252,18.462,15.123,3.833,4.294,3.670,36.0,13.0,13.0,62.0,3.0,3.0,0.0,5.0,0.0,18.0,20.0,33.0,46.0,8.0,19.0,29.0,20.0,9.0,6.0,30.0,13.0,6.0,1.092,1.703,1.662,0.0,2.0,1.0,57.921,49.0,46.481,2.0,0.626,0.954,0.954,4.128,3.953,3.584,0,1,-1.0,-1.0,-8.0,0.0
2,NYR,2022,NYR,2022020023,NYR,WPG,AWAY,20221014,Team Level,all,0.4000,0.5185,0.5481,3600.0,40.542,2.814,2.906,9.682,1.407,23.631,16.561,2.725,2.863,2.773,41.0,16.0,13.0,70.0,1.0,6.0,0.0,7.0,2.0,21.0,20.0,40.0,56.0,3.0,6.0,28.0,25.0,3.0,20.0,46.0,10.0,1.0,1.318,1.288,0.208,1.0,0.0,0.0,69.733,57.0,56.422,18.0,0.645,0.430,0.430,3.029,3.073,2.982,33.758,4.221,2.192,7.402,1.079,18.976,13.131,4.109,4.173,4.062,3

## Rolling average features:

In [71]:
# Sort by team and date first
df_games = df_all.sort_values(['playerTeam', 'gameDate']).reset_index(drop=True)

# Average for a team's last 3 games average (PRE-GAME: shift(1))
df_games['xGoalsFor_last3'] = (
    df_games.groupby('playerTeam')['xGoalsFor']
    .apply(lambda s: s.shift(1).rolling(window=3).mean())
    .reset_index(0, drop=True)
)
df_games['shotsOnGoalFor_last3'] = (
    df_games.groupby('playerTeam')['shotsOnGoalFor']
    .apply(lambda s: s.shift(1).rolling(window=3).mean())
    .reset_index(0, drop=True)
)
df_games['xGoalsAgainst_last3'] = (
    df_games.groupby('playerTeam')['xGoalsAgainst']
    .apply(lambda s: s.shift(1).rolling(window=3).mean())
    .reset_index(0, drop=True)
)

# For a team's last 5 games average (PRE-GAME: shift(1))
df_games['xGoalsFor_last5'] = (
    df_games.groupby('playerTeam')['xGoalsFor']
    .apply(lambda s: s.shift(1).rolling(window=5).mean())
    .reset_index(0, drop=True)
)
df_games['shotsOnGoalFor_last5'] = (
    df_games.groupby('playerTeam')['shotsOnGoalFor']
    .apply(lambda s: s.shift(1).rolling(window=5).mean())
    .reset_index(0, drop=True)
)
df_games['xGoalsAgainst_last5'] = (
    df_games.groupby('playerTeam')['xGoalsAgainst']
    .apply(lambda s: s.shift(1).rolling(window=5).mean())
    .reset_index(0, drop=True)
)

# For a team's last 10 games average (PRE-GAME: shift(1))
df_games['xGoalsFor_last10'] = (
    df_games.groupby('playerTeam')['xGoalsFor']
    .apply(lambda s: s.shift(1).rolling(window=10).mean())
    .reset_index(0, drop=True)
)
df_games['shotsOnGoalFor_last10'] = (
    df_games.groupby('playerTeam')['shotsOnGoalFor']
    .apply(lambda s: s.shift(1).rolling(window=10).mean())
    .reset_index(0, drop=True)
)
df_games['xGoalsAgainst_last10'] = (
    df_games.groupby('playerTeam')['xGoalsAgainst']
    .apply(lambda s: s.shift(1).rolling(window=10).mean())
    .reset_index(0, drop=True)
)

# For season so far (PRE-GAME: shift(1) then expanding)
df_games['xGoalsFor_season'] = (
    df_games.groupby(['playerTeam', 'season'])['xGoalsFor']
    .apply(lambda s: s.shift(1).expanding().mean())
    .reset_index([0, 1], drop=True)
)
df_games['xshotsOnGoalFor_season'] = (
    df_games.groupby(['playerTeam', 'season'])['shotsOnGoalFor']
    .apply(lambda s: s.shift(1).expanding().mean())
    .reset_index([0, 1], drop=True)
)
df_games['xGoalsAgainst_season'] = (
    df_games.groupby(['playerTeam', 'season'])['xGoalsAgainst']
    .apply(lambda s: s.shift(1).expanding().mean())
    .reset_index([0, 1], drop=True)
)

# Function to calculate slope for a rolling window
def calculate_slope(series):
    if len(series) < 2 or series.isna().any():
        return np.nan
    x = np.arange(len(series))  # Days/games (0, 1, 2, ...)
    y = series.values
    # Calculate slope using linear regression formula
    slope = np.polyfit(x, y, 1)[0]
    return slope

# Calculate slopes for last 5 games (PRE-GAME: shift(1) then rolling)
df_games['xGoalsFor_slope5'] = (
    df_games.groupby('playerTeam')['xGoalsFor']
    .apply(lambda s: s.shift(1).rolling(window=5).apply(calculate_slope, raw=False))
    .reset_index(0, drop=True)
)

df_games['xGoalsAgainst_slope5'] = (
    df_games.groupby('playerTeam')['xGoalsAgainst']
    .apply(lambda s: s.shift(1).rolling(window=5).apply(calculate_slope, raw=False))
    .reset_index(0, drop=True)
)

df_games['shotsOnGoalFor_slope5'] = (
    df_games.groupby('playerTeam')['shotsOnGoalFor']
    .apply(lambda s: s.shift(1).rolling(window=5).apply(calculate_slope, raw=False))
    .reset_index(0, drop=True)
)

df_games['shotsOnGoalAgainst_slope5'] = (
    df_games.groupby('playerTeam')['shotsOnGoalAgainst']
    .apply(lambda s: s.shift(1).rolling(window=5).apply(calculate_slope, raw=False))
    .reset_index(0, drop=True)
)

df_games.head()

df_games.head(10)
print(df_games.columns.tolist())


['team', 'season', 'name', 'gameId', 'playerTeam', 'opposingTeam', 'home_or_away', 'gameDate', 'position', 'situation', 'xGoalsPercentage', 'corsiPercentage', 'fenwickPercentage', 'iceTime', 'xOnGoalFor', 'xGoalsFor', 'xReboundsFor', 'xFreezeFor', 'xPlayStoppedFor', 'xPlayContinuedInZoneFor', 'xPlayContinuedOutsideZoneFor', 'flurryAdjustedxGoalsFor', 'scoreVenueAdjustedxGoalsFor', 'flurryScoreVenueAdjustedxGoalsFor', 'shotsOnGoalFor', 'missedShotsFor', 'blockedShotAttemptsFor', 'shotAttemptsFor', 'goalsFor', 'reboundsFor', 'reboundGoalsFor', 'freezeFor', 'playStoppedFor', 'playContinuedInZoneFor', 'playContinuedOutsideZoneFor', 'savedShotsOnGoalFor', 'savedUnblockedShotAttemptsFor', 'penaltiesFor', 'penalityMinutesFor', 'faceOffsWonFor', 'hitsFor', 'takeawaysFor', 'giveawaysFor', 'lowDangerShotsFor', 'mediumDangerShotsFor', 'highDangerShotsFor', 'lowDangerxGoalsFor', 'mediumDangerxGoalsFor', 'highDangerxGoalsFor', 'lowDangerGoalsFor', 'mediumDangerGoalsFor', 'highDangerGoalsFor', 'scor

In [72]:
# Start with your current df that has rolling averages and momentum features
df_matchup = df_games.copy()

# Make sure it's sorted by game and team
df_matchup = df_matchup.sort_values(['gameId', 'playerTeam'])

# Separate home and away teams
home_teams = df_matchup[df_matchup['home_or_away'] == 'HOME'].copy()
away_teams = df_matchup[df_matchup['home_or_away'] == 'AWAY'].copy()

# Rename columns to distinguish team vs opponent
home_cols = {col: f'home_{col}' for col in home_teams.columns if col not in ['gameId', 'gameDate']}
away_cols = {col: f'away_{col}' for col in away_teams.columns if col not in ['gameId', 'gameDate']}

home_teams = home_teams.rename(columns=home_cols)
away_teams = away_teams.rename(columns=away_cols)

# Merge on gameId so each row = one game with both teams
df_games = pd.merge(
    home_teams,
    away_teams,
    on='gameId',
    suffixes=('_home', '_away')
)

print("Games dataframe shape:", df_games.shape)
df_games.head()
print([col for col in df_games.columns if 'home' in col or 'away' in col][:10])

Games dataframe shape: (4124, 263)
['home_team', 'home_season', 'home_name', 'home_playerTeam', 'home_opposingTeam', 'home_home_or_away', 'gameDate_home', 'home_position', 'home_situation', 'home_xGoalsPercentage']


In [73]:
# List all rolling average and momentum features
rolling_features = [
    'xGoalsFor_last5',
    'xGoalsFor_last10', 
    'xGoalsFor_season',
    'xGoalsAgainst_last5',
    'xGoalsAgainst_last10',
    'xGoalsAgainst_season',
    'xGoalsFor_slope5',
    'xGoalsAgainst_slope5',
    'shotsOnGoalFor_last5',
    'shotsOnGoalAgainst_last5',
    'shotsOnGoalFor_slope5',
    'shotsOnGoalAgainst_slope5',
]

# Create home - away differentials
for feature in rolling_features:
    home_col = f'home_{feature}'
    away_col = f'away_{feature}'
    
    if home_col in df_games.columns and away_col in df_games.columns:
        df_games[f'{feature}_diff'] = df_games[home_col] - df_games[away_col]
    else:
        print(f"Warning: {feature} not found in both home and away")

# Check
diff_cols = [col for col in df_games.columns if '_diff' in col]
print(f"\nCreated {len(diff_cols)} differential features:")
print(diff_cols)


Created 19 differential features:
['home_shot_diff', 'home_penalty_diff', 'home_shotAttempt_diff', 'home_faceOff_diff', 'away_shot_diff', 'away_penalty_diff', 'away_shotAttempt_diff', 'away_faceOff_diff', 'xGoalsFor_last5_diff', 'xGoalsFor_last10_diff', 'xGoalsFor_season_diff', 'xGoalsAgainst_last5_diff', 'xGoalsAgainst_last10_diff', 'xGoalsAgainst_season_diff', 'xGoalsFor_slope5_diff', 'xGoalsAgainst_slope5_diff', 'shotsOnGoalFor_last5_diff', 'shotsOnGoalFor_slope5_diff', 'shotsOnGoalAgainst_slope5_diff']


## Prepare features and target

In [74]:
X = df_games[['xGoalsFor_last5_diff',
       'xGoalsFor_last10_diff', 'xGoalsFor_season_diff',
       'xGoalsAgainst_last5_diff', 'xGoalsAgainst_last10_diff',
       'xGoalsAgainst_season_diff', 'xGoalsFor_slope5_diff',
       'xGoalsAgainst_slope5_diff', 'shotsOnGoalFor_last5_diff',
       'shotsOnGoalFor_slope5_diff', 'shotsOnGoalAgainst_slope5_diff']]
X_matchup = X

# Target: did home team win?
y_matchup = df_games['home_win']

# Drop any rows with missing values
valid_idx = X_matchup.dropna().index
X_matchup = X_matchup.loc[valid_idx]
y_matchup = y_matchup.loc[valid_idx]

print(f"Final dataset shape: {X_matchup.shape}")

Final dataset shape: (3885, 11)


In [75]:
# Count wins and losses
win_count = (df_games['home_win'] == 1).astype(int).sum()
loss_count = (df_games['away_win'] == 1).astype(int).sum()
print(f"Wins: {win_count}, Losses: {loss_count}")
print(f"Home advantage winning percentage: {win_count / (win_count + loss_count) * 100:.2f}%")

Wins: 2240, Losses: 1884
Home advantage winning percentage: 54.32%


In [76]:
# Sort by date
df_matchups_sorted = df_games.loc[valid_idx].sort_values('gameId')

# 80/20 chronological split
split_idx = int(len(X_matchup) * 0.8)

X_train = X_matchup.iloc[:split_idx]
X_test = X_matchup.iloc[split_idx:]
y_train = y_matchup.iloc[:split_idx]
y_test = y_matchup.iloc[split_idx:]

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

Train size: (3108, 11)
Test size: (777, 11)


In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42
)
y_train_ravel = y_train.values.ravel()
y_test_ravel = y_test.values.ravel()

model.fit(X_train, y_train_ravel)

train_accuracy = model.score(X_train, y_train_ravel)
test_accuracy = model.score(X_test, y_test_ravel)

print(f"\nMatchup Differential Approach:")
print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(X_train.columns)


Matchup Differential Approach:
Train accuracy: 0.6393
Test accuracy: 0.5753
Index(['xGoalsFor_last5_diff', 'xGoalsFor_last10_diff',
       'xGoalsFor_season_diff', 'xGoalsAgainst_last5_diff',
       'xGoalsAgainst_last10_diff', 'xGoalsAgainst_season_diff',
       'xGoalsFor_slope5_diff', 'xGoalsAgainst_slope5_diff',
       'shotsOnGoalFor_last5_diff', 'shotsOnGoalFor_slope5_diff',
       'shotsOnGoalAgainst_slope5_diff'],
      dtype='object')
